# portfolioSR: analisi degli artifact

Eseguire prima una baseline dalla root del repository:

```bash
poetry install --with experiments,notebooks
poetry run python -m qreduce.experiments.portfolio_sr --run-id sr_baseline --asset-count 20 --max-reduction-rounds 0
```

Questo notebook carica soltanto risultati già persistiti. Cambiare `run_directory`
per analizzare una riduzione o uno screening. Le metriche mancanti non sono zeri.

`position_count` conta i pesi investiti sopra la tolleranza. Quando disponibile,
`selected_count` conta invece le binarie di selezione attive, anche a peso zero.
I nuovi modelli hanno sempre binarie, anche nella baseline senza cardinalità.
Negli artifact storici senza binarie il conteggio non è registrato e non si deduce dai pesi.
Selezioni a peso zero possono variare a parità di pesi e Sharpe. Byte identici di
`problem.json` non dimostrano che run di versioni diverse abbiano risolto lo stesso modello.


In [ ]:
from dataclasses import asdict
from pathlib import Path

from qreduce.experiments.portfolio_sr.analysis import (
    extract_portfolio_weights,
    extract_sector_exposures,
    load_sharpe_portfolio_analysis,
)
from qreduce.notebooks.presentation import display_vertical

repository_root = next(
    directory for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / "pyproject.toml").is_file()
)
run_directory = repository_root / "outputs/portfolio_sr/sr_baseline"
analysis = load_sharpe_portfolio_analysis(run_directory)


In [ ]:
display_vertical(analysis.artifacts.manifest, title="Manifest")
display_vertical(analysis.artifacts.result.get("run_result", {}), title="Stato e diagnostica")
display_vertical(analysis.problem_summary, title="Problema originale e dominio residuo")
display_vertical(analysis.artifacts.fixing_manifest or {}, title="Fixing e provenance")


In [ ]:
display_vertical(asdict(analysis.solution) if analysis.solution else {}, title="Metriche adottate")
display_vertical(asdict(analysis.reduction) if analysis.reduction else {}, title="Riduzione")
display_vertical(asdict(analysis.timing) if analysis.timing else {}, title="Tempi")


## Pesi ed esposizioni

I pesi sotto soglia sono filtrati soltanto per la presentazione, senza
rinormalizzazione. Le esposizioni comprendono i settori a peso zero e usano
l'intero portafoglio. Soluzioni diagnostiche o incomplete non sono presentate.


In [ ]:
display_vertical(extract_portfolio_weights(analysis.artifacts), title="Allocazioni")
display_vertical(extract_sector_exposures(analysis.artifacts), title="Esposizioni settoriali")
